In [ ]:
# from jupytergis import GISDocument
from jupytergis.tiler import GISDocument
from typing import Iterable, List
from pyproj import Transformer
import pystac_client
import stackstac

def extent_3857_to_bbox_4326(extent: Iterable[float]) -> List[float]:
    """
    Convert a Web Mercator extent to a STAC-ready lon/lat bbox.
    Input extent format: [minx, miny, maxx, maxy] in EPSG:3857
    Output bbox format:  [min_lon, min_lat, max_lon, max_lat] in EPSG:4326
    """
    minx, miny, maxx, maxy = extent
    transformer = Transformer.from_crs(
        "EPSG:3857",
        "EPSG:4326",
        always_xy=True,  # keep x=lon, y=lat ordering
    )
    min_lon, min_lat = transformer.transform(minx, miny)
    max_lon, max_lat = transformer.transform(maxx, maxy)
    return [min_lon, min_lat, max_lon, max_lat]

doc = GISDocument("footprints.jGIS")

In [ ]:
m = doc.extent
# Full map extent in lon/lat (for reference only).
bbox_doc = (
    extent_3857_to_bbox_4326([m.west, m.south, m.east, m.north]) if m else None
)
# STAC search and stackstac.stack must use the *same* bounds; otherwise the
# stack is huge and mostly empty while items only cover a small area.
bbox_search = [ -18.248776440117897, 27.004514659408795, -16.033094271404384, 34.01156206721229 ]
# [
#     -4.85,   # west  (min lon)
#     47.56,    # south (min lat)
#     -3.99,   # east  (max lon)
#     47.93,     # north (max lat)
# ] 
# [-2.98, 47.50, -2.58, 47.66]

bbox = bbox_search
bbox_doc, bbox

In [ ]:
# URL = "https://stac.dataspace.copernicus.eu/v1"
URL = "https://earth-search.aws.element84.com/v1"
catalog = pystac_client.Client.open(URL)

In [ ]:
items = catalog.search(
    bbox=bbox,
    collections=["sentinel-2-l2a"],
    datetime="2020-03-01/2020-03-02",
).item_collection()
len(items)

In [ ]:
da = stackstac.stack(items, epsg=4326, bounds_latlon=bbox)

In [ ]:
state = {"lazy": None, "computed": None}

def ndvi_process(sliced):
    red = sliced.sel(band='red')
    nir = sliced.sel(band='nir')
    out = (nir - red) / (nir + red)
    out = out.where((nir + red) != 0)
    out.name = 'ndvi'
    return out

def ndwi_process(sliced):
    green = sliced.sel(band='green')
    nir = sliced.sel(band='nir')
    ndwi = (green - nir) / (green + nir)
    ndwi = ndwi.where((green + nir) != 0)
    ndwi.name = 'ndwi'
    return ndwi

sub_id = doc.bind_extent_slice_callback(
    da,
    on_slice=lambda out: state.update({"lazy": out}),  # lazy dask-backed DataArray
    data_crs='EPSG:4326',
    x_name='x',
    y_name='y',
    process=ndwi_process,
    compute=False,  # <-- keep lazy
    strict_process=False,  # keep state updates even if process throws
)

In [ ]:
doc

In [ ]:
state["lazy"]

In [ ]:
import dask.diagnostics

lazy = state["lazy"]
assert lazy is not None, "Pan/zoom the map (or set extent) so the callback runs."
with dask.diagnostics.ProgressBar():
    state["computed"] = lazy.compute()


In [ ]:
import math
import numpy as np

c = state["computed"]
if "time" in c.dims:
    finite_fracs = []
    for t in range(c.sizes["time"]):
        v = c.isel(time=t).values
        finite_fracs.append(float(np.isfinite(v).mean()))
    best_t = int(np.argmax(finite_fracs))
    img = c.isel(time=best_t)
else:
    best_t = None
    img = c

vals = img.values
finite = np.isfinite(vals)
if finite.any():
    vmin = float(np.nanpercentile(vals, 2))
    vmax = float(np.nanpercentile(vals, 98))
else:
    vmin, vmax = -0.5, 0.5

if (not math.isfinite(vmin)) or (not math.isfinite(vmax)) or (vmin >= vmax):
    vmin, vmax = -0.5, 0.5

(best_t, img.dims, img.shape, (vmin, vmax))


In [ ]:
await doc.add_tiler_layer(
    name='ndwi-manual-compute',
    data_array=img,
    colormap_name='plasma',
    rescale=(vmin, vmax),
    reproject='max',
)

In [ ]:
doc.unbind_extent_slice_callback(sub_id)
